# 01. Raw read QC（Quality Control）

このノートブックは、FASTQが解析に耐えられるかを見る段階である。

**この段階が答える問い**

- FASTQファイルが壊れていないか。
- read数がサンプル間で極端に違わないか。
- read quality、adapter contamination、GC biasに大きな問題がないか。

**入力**

- `metadata/samples.tsv`
- `raw_data/` のFASTQファイル

**出力**

- `results/qc/fastq_inventory.tsv`
- `results/qc/fastqc/`
- `results/qc/multiqc_report.html`

**次に進む条件**

- ファイル破損がない。
- FastQC/MultiQCで重大な問題がない、または問題を理解して次に進む判断ができる。


In [ ]:
from pathlib import Path
import gzip
import hashlib
import json
import shutil
import subprocess

import pandas as pd

PROJECT_DIR = Path("/Users/yusuke_tateishi/Documents/RNA_seq").resolve()
CONFIG = json.loads((PROJECT_DIR / "config" / "analysis_config.json").read_text(encoding="utf-8"))
SAMPLES = pd.read_csv(PROJECT_DIR / CONFIG["samples_path"], sep="\t")
QC_DIR = PROJECT_DIR / "results" / "qc"
FASTQC_DIR = QC_DIR / "fastqc"
QC_DIR.mkdir(parents=True, exist_ok=True)
FASTQC_DIR.mkdir(parents=True, exist_ok=True)

SAMPLES


## FASTQ inventory

まず、ファイルサイズと存在確認をする。これは品質評価ではなく、解析対象のファイル一覧を固定する作業である。


In [ ]:
rows = []
for _, sample in SAMPLES.iterrows():
    for mate_column in ["fastq_1", "fastq_2"]:
        path = PROJECT_DIR / sample[mate_column]
        rows.append(
            {
                "sample_id": sample["sample_id"],
                "mate": mate_column,
                "path": str(path.relative_to(PROJECT_DIR)),
                "exists": path.exists(),
                "size_gb": path.stat().st_size / 1024**3 if path.exists() else None,
            }
        )

inventory = pd.DataFrame(rows)
inventory.to_csv(QC_DIR / "fastq_inventory.tsv", sep="\t", index=False)
inventory


## gzipとして読めるか確認する

`.fq.gz` はgzip圧縮ファイルである。先頭1行だけ読んで、圧縮ファイルとして壊れていないかを軽く確認する。


In [ ]:
RUN_GZIP_SMOKE_TEST = True

if RUN_GZIP_SMOKE_TEST:
    gzip_rows = []
    for path_text in inventory["path"]:
        path = PROJECT_DIR / path_text
        try:
            with gzip.open(path, "rt") as handle:
                first_line = handle.readline().strip()
            gzip_rows.append({"path": path_text, "gzip_readable": True, "first_line": first_line[:80]})
        except Exception as exc:
            gzip_rows.append({"path": path_text, "gzip_readable": False, "first_line": str(exc)})
    gzip_check = pd.DataFrame(gzip_rows)
    gzip_check.to_csv(QC_DIR / "gzip_smoke_test.tsv", sep="\t", index=False)
    display(gzip_check)


## read数を数える

FASTQは4行で1 readである。行数を4で割るとread数になる。大きいファイルなので時間がかかる場合がある。


In [ ]:
RUN_READ_COUNTS = False

def count_fastq_reads(path: Path) -> int:
    line_count = 0
    with gzip.open(path, "rt") as handle:
        for line_count, _ in enumerate(handle, start=1):
            pass
    return line_count // 4

if RUN_READ_COUNTS:
    read_rows = []
    for _, sample in SAMPLES.iterrows():
        r1 = count_fastq_reads(PROJECT_DIR / sample["fastq_1"])
        r2 = count_fastq_reads(PROJECT_DIR / sample["fastq_2"])
        read_rows.append({"sample_id": sample["sample_id"], "reads_R1": r1, "reads_R2": r2})
    read_counts = pd.DataFrame(read_rows)
    read_counts.to_csv(QC_DIR / "read_counts.tsv", sep="\t", index=False)
    display(read_counts)
else:
    print("Set RUN_READ_COUNTS = True to count reads. This can take time.")


## FastQCを実行する

FastQCは各FASTQを個別に検査する。結果はHTMLとzipで出る。

初回だけ `RUN_FASTQC = True` にしよう。


In [ ]:
RUN_FASTQC = False
THREADS = int(CONFIG["quantification"].get("threads", 8))

fastq_paths = [str(PROJECT_DIR / path) for path in inventory["path"]]
fastqc_command = ["fastqc", "--threads", str(THREADS), "--outdir", str(FASTQC_DIR), *fastq_paths]
(QC_DIR / "fastqc_command.txt").write_text(" ".join(fastqc_command) + "\n", encoding="utf-8")

if RUN_FASTQC:
    fastqc = shutil.which("fastqc")
    if fastqc is None:
        raise RuntimeError("fastqc was not found. Activate the rna-seq environment first.")
    fastqc_command[0] = fastqc
    subprocess.run(fastqc_command, check=True)
else:
    print("Set RUN_FASTQC = True to run FastQC.")
    print("Command saved to:", (QC_DIR / "fastqc_command.txt").relative_to(PROJECT_DIR))


## MultiQCを実行する

MultiQCは、複数サンプルのFastQC結果を1つのHTMLにまとめる。FastQCが終わってから実行する。


In [ ]:
RUN_MULTIQC = False

if RUN_MULTIQC:
    multiqc = shutil.which("multiqc")
    if multiqc is None:
        raise RuntimeError("multiqc was not found. Activate the rna-seq environment first.")
    command = [multiqc, str(FASTQC_DIR), "--outdir", str(QC_DIR), "--filename", "multiqc_report.html"]
    subprocess.run(command, check=True)
else:
    print("Set RUN_MULTIQC = True after FastQC finishes.")


## 何を見ればよいか

MultiQCでまず見る項目は次の通りである。

- Per base sequence quality: 末端で少し下がる程度ならよくある。
- Adapter content: 強く出る場合はtrimmingを検討する。
- GC content: サンプル間で大きくずれる場合は要注意である。
- Sequence duplication levels: ライブラリの偏りや高発現RNAの影響を見る。
- Total sequences: サンプル間で極端に違わないかを見る。

**小さい練習**

`results/qc/fastq_inventory.tsv` を開いて、R1/R2のペア数が各サンプルでそろっているか確認しよう。
